In [0]:
WITH active_members AS (
    SELECT cde_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE snapshot_date_id = (SELECT MAX(snapshot_date_id)
                              FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot)
      AND renewal_status LIKE 'Active%'
),
scene_ticket_buyers AS (
    SELECT
        t.CDE_ID,
        SUM(t.Quantity) AS net_tickets,
        SUM(CASE WHEN f.language IN ('Hindi','Punjabi','Gujarati','Telugu','Tamil')
                 THEN t.Quantity ELSE 0 END) AS net_indian_tickets
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_film f
      ON t.film_id = f.film_id
    WHERE t.TicketFLAG = 'Yes'
      AND t.SCENEFLAG = 'Yes'
      AND t.VisitDateId >= CAST(date_format(add_months(current_date(), -24), 'yyyyMMdd') AS INT)
      AND t.VisitDateId <  CAST(date_format(current_date(), 'yyyyMMdd') AS INT)Diwa
      AND t.CDE_ID > 0
    GROUP BY t.CDE_ID
    HAVING SUM(t.Quantity) > 0          -- refunds netted out
),
scene_non_cc AS (
    SELECT s.*
    FROM scene_ticket_buyers s
    LEFT ANTI JOIN active_members a
      ON s.CDE_ID = a.cde_id
)
SELECT
    COUNT(*)                                                    AS scene_non_cc_moviegoers,
    SUM(CASE WHEN net_indian_tickets > 0 THEN 1 ELSE 0 END)    AS watched_indian_film_p12m,
    ROUND(SUM(CASE WHEN net_indian_tickets > 0 THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) AS pct
FROM scene_non_cc;